# Pipe line of the UNet model for the binary classification of the annual LiDAR data

## Importing the necessary libraries

In [70]:
import os
from osgeo import gdal
import os
from sklearn.metrics import accuracy_score, f1_score, jaccard_score, precision_score, recall_score
import matplotlib.pyplot as plt
import SinkTools2 as st
import tqdm
import torch
from osgeo import gdal
import numpy as np
import torch.nn.functional as F
from model import build_unet
import time
import SinkTools2 as st

## Define all necessary inputs

In [67]:
start_time = time.time()
input_raster_path = '/Users/nirgut/Downloads/pipeline_test/lidar.tif'
output_tiles_dir = '/Users/nirgut/Downloads/pipeline_test/tiles/'
output_pred_dir = '/Users/nirgut/Downloads/pipeline_test/pred_tiles/'
# combined_pred_path = '/Users/nirgut/Downloads/pipeline_test/combined_pred.tif'
output_mosaic = '/Users/nirgut/Downloads/pipeline_test/combined_pred_new.tif'
output_shapefile = '/Users/nirgut/Downloads/pipeline_test/output_shapefile.shp'

model_path = '/Users/nirgut/ML_Course/2nd_unet/files/no_overlap_epoch_54.pth'

tile_size_x = 256  # width of each tile in pixels
tile_size_y = 256  # height of each tile in pixels


## importing the raster data and slicing it to even tiles

In [38]:
# import os
# import numpy as np
# from osgeo import gdal
# 
# def normalize_array(array):
#     """Normalize the array values to the range [0, 1]."""
#     array_min = array.min()
#     array_max = array.max()
#     if array_max - array_min == 0:
#         return array  # Avoid division by zero if all values are the same
#     normalized_array = (array - array_min) / (array_max - array_min)
#     return normalized_array
# 
# def divide_raster_to_tiles(input_raster_path, output_dir, tile_size_x, tile_size_y):
#     # Open the input raster
#     raster = gdal.Open(input_raster_path)
#     if raster is None:
#         print(f"Unable to open input raster {input_raster_path}")
#         return
# 
#     # Get raster information
#     raster_band = raster.GetRasterBand(1)
#     xsize = raster.RasterXSize
#     ysize = raster.RasterYSize
#     geotransform = raster.GetGeoTransform()
#     projection = raster.GetProjection()
# 
#     # Calculate the number of tiles in x and y directions
#     x_tiles = (xsize + tile_size_x - 1) // tile_size_x
#     y_tiles = (ysize + tile_size_y - 1) // tile_size_y
# 
#     # Create output directory if it doesn't exist
#     if not os.path.exists(output_dir):
#         os.makedirs(output_dir)
# 
#     for i in range(x_tiles):
#         for j in range(y_tiles):
#             # Calculate the offsets and tile size
#             x_offset = i * tile_size_x
#             y_offset = j * tile_size_y
#             x_tile_size = min(tile_size_x, xsize - x_offset)
#             y_tile_size = min(tile_size_y, ysize - y_offset)
# 
#             # Read the tile data
#             tile = raster_band.ReadAsArray(x_offset, y_offset, x_tile_size, y_tile_size)
# 
#             # Normalize the tile data
#             tile = normalize_array(tile)
# 
#             # Create a new raster dataset for the tile
#             driver = gdal.GetDriverByName('GTiff')
#             output_tile_path = os.path.join(output_dir, f'tile_{i}_{j}.tif')
#             tile_dataset = driver.Create(output_tile_path, x_tile_size, y_tile_size, 1, gdal.GDT_Float32)
#             tile_dataset.SetGeoTransform((
#                 geotransform[0] + x_offset * geotransform[1],
#                 geotransform[1],
#                 0,
#                 geotransform[3] + y_offset * geotransform[5],
#                 0,
#                 geotransform[5]
#             ))
#             tile_dataset.SetProjection(projection)
#             tile_dataset.GetRasterBand(1).WriteArray(tile)
# 
#             # Save and close the tile dataset
#             tile_dataset.FlushCache()
#             tile_dataset = None
# 
#     # Close the original raster
#     raster = None



In [39]:
# divide_raster_to_tiles(input_raster_path, output_tiles_dir, tile_size_x, tile_size_y)

In [40]:
def normalize_array(array):
    """Normalize the array values to the range [0, 1]."""
    array_min = array.min()
    array_max = array.max()
    if array_max - array_min == 0:
        return array  # Avoid division by zero if all values are the same
    normalized_array = (array - array_min) / (array_max - array_min)
    return normalized_array

def divide_raster_to_tiles(input_raster_path, output_dir, tile_size_x, tile_size_y, overlap_x=0, overlap_y=0):
    # Open the input raster
    raster = gdal.Open(input_raster_path)
    if raster is None:
        print(f"Unable to open input raster {input_raster_path}")
        return

    # Get raster information
    raster_band = raster.GetRasterBand(1)
    xsize = raster.RasterXSize
    ysize = raster.RasterYSize
    geotransform = raster.GetGeoTransform()
    projection = raster.GetProjection()

    # Calculate the number of tiles in x and y directions, considering overlap
    x_tiles = (xsize + tile_size_x - 1) // (tile_size_x - overlap_x)
    y_tiles = (ysize + tile_size_y - 1) // (tile_size_y - overlap_y)

    # Create output directory if it doesn't exist
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    for i in range(x_tiles):
        for j in range(y_tiles):
            # Calculate the offsets and tile size
            x_offset = i * (tile_size_x - overlap_x)
            y_offset = j * (tile_size_y - overlap_y)
            x_tile_size = min(tile_size_x, xsize - x_offset)
            y_tile_size = min(tile_size_y, ysize - y_offset)

            # Ensure that the tile dimensions are positive
            if x_tile_size <= 0 or y_tile_size <= 0:
                continue

            # Read the tile data
            tile = raster_band.ReadAsArray(x_offset, y_offset, x_tile_size, y_tile_size)

            # Normalize the tile data
            tile = normalize_array(tile)

            # Create a new raster dataset for the tile
            driver = gdal.GetDriverByName('GTiff')
            output_tile_path = os.path.join(output_dir, f'tile_{i}_{j}.tif')
            tile_dataset = driver.Create(output_tile_path, x_tile_size, y_tile_size, 1, gdal.GDT_Float32)
            tile_dataset.SetGeoTransform((
                geotransform[0] + x_offset * geotransform[1],
                geotransform[1],
                0,
                geotransform[3] + y_offset * geotransform[5],
                0,
                geotransform[5]
            ))
            tile_dataset.SetProjection(projection)
            tile_dataset.GetRasterBand(1).WriteArray(tile)

            # Save and close the tile dataset
            tile_dataset.FlushCache()
            tile_dataset = None

    # Close the original raster
    raster = None

In [41]:
overlap_x = 100  # Example overlap in x direction
overlap_y = 100  # Example overlap in y direction

divide_raster_to_tiles(input_raster_path, output_tiles_dir, tile_size_x, tile_size_y, overlap_x, overlap_y)

# Prediction for each tile
## using the pre-trained UNet model to predict the tiles, and save the results as binary tiles

In [42]:
threshold = 0.5
save_array = False
calc_each_samp = False

In [43]:
def predict_mask(raster_file, checkpoint_path, device):
    """
    The predict_mask function loads a pre-trained U-Net model from a checkpoint file, processes a raster image to the appropriate input size, performs a forward pass through the model to predict a mask, and returns a binary mask indicating the areas of interest.
    :param raster_file: the tif tile file of DEM
    :param checkpoint_path: saved model path
    :param device: cuda/cpu
    :return: 
    """
    """ Load model """
    model = build_unet()
    model = model.to(device)
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.eval()

    """ Reading raster """
    raster = gdal.Open(raster_file)
    raster = raster.ReadAsArray()

    """ Resize raster """
    new_height, new_width = 256, 256
    raster = raster.astype(np.float32)
    raster = torch.from_numpy(raster)
    raster = raster.unsqueeze(0)  # Add batch dimension
    raster = F.interpolate(raster.unsqueeze(0), size=(new_height, new_width), mode='bilinear', align_corners=False)

    """ Prediction """
    with torch.no_grad():
        pred_mask = model(raster)
        pred_mask = torch.sigmoid(pred_mask)
        pred_mask = pred_mask.cpu().numpy()
        pred_mask = np.squeeze(pred_mask, axis=0)  # Remove batch dimension
        # make a mask of 0s and 1s
        pred_mask = np.where(pred_mask >= threshold, 1, 0)

    # print(f'pred_mask shape: {pred_mask.shape}')

    return pred_mask

In [44]:
# save pred_mask as a binary raster
def save_mask_as_raster(pred_mask, raster_file, output_file):
    """
    The save_mask_as_raster function saves a binary mask as a raster file.
    :param pred_mask: the binary mask
    :param raster_file: the tif tile file of DEM
    :param output_file: the output file path
    :return:
    """
    raster = gdal.Open(raster_file)
    driver = gdal.GetDriverByName('GTiff')
    output_raster = driver.Create(output_file, raster.RasterXSize, raster.RasterYSize, 1, gdal.GDT_Byte)
    output_raster.SetGeoTransform(raster.GetGeoTransform())
    output_raster.SetProjection(raster.GetProjection())
    # output_raster.GetRasterBand(1).WriteArray(pred_mask)
    output_raster.FlushCache()
    output_raster = None

In [45]:
def array2raster(raster_path, array, output_path, geotransform, projection):
    """
    Save a NumPy array as a raster file using GDAL.

    Parameters:
        raster_path (str): Path to the original raster file to use as a template.
        array (numpy.ndarray): NumPy array containing the data to be saved as a raster.
        output_path (str): Path to save the raster file.
        geotransform (tuple): Geotransform parameters (top left x, w-e pixel resolution, rotation, top left y, rotation, n-s pixel resolution).
        projection (str): Projection in WKT format.
    """
    # Open the original raster file
    original_raster = gdal.Open(raster_path)

    # Get the number of rows and columns from the original raster
    array = np.squeeze(array)
    # print(f'array shape: {array.shape}')
    rows, cols = array.shape

    # Create the output raster file
    driver = gdal.GetDriverByName('GTiff')
    out_raster = driver.Create(output_path, cols, rows, 1, gdal.GDT_Float32)

    # Write the array data to the output raster
    out_band = out_raster.GetRasterBand(1)
    out_band.WriteArray(array)

    # Set the geotransform and projection
    out_raster.SetGeoTransform(geotransform)
    out_raster.SetProjection(projection)

    # Close the output raster file
    out_band.FlushCache()

In [46]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Predicting the mask for each tile

In [47]:
for tile_file in tqdm.tqdm(os.listdir(output_tiles_dir)):
    if tile_file.endswith('.tif'):
        # print(tile_file)
        tile_path = os.path.join(output_tiles_dir, tile_file)
        pred_mask = predict_mask(tile_path, model_path, device)
        pred_mask = pred_mask.squeeze()
        output_file = os.path.join(output_pred_dir, tile_file)
        # save_mask_as_raster(pred_mask, tile_path, output_file)
        array2raster(tile_path, pred_mask, output_file, gdal.Open(tile_path).GetGeoTransform(), gdal.Open(tile_path).GetProjection()) 

100%|██████████| 210/210 [04:16<00:00,  1.22s/it]


## Combining the tiles to a single raster based on the original raster

In [65]:
import os
import subprocess

def combine_rasters_with_gdal(folder_path, output_file):
    # List all .tif files in the folder
    input_files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith('.tif')]
    
    # Build the gdal_merge command
    gdal_merge_command = [
        'gdal_merge.py',
        '-o', output_file,
        '-n', '0',  # NoData value to ignore
        '-a_nodata', '0',  # NoData value to set in the output
        '-pct',  # Use the maximum value in overlapping regions
    ] + input_files
    
    # Execute the command
    subprocess.run(gdal_merge_command)

ERROR 6: /Users/nirgut/Downloads/pipeline_test/combined_pred_new.tif, band 1: SetColorTable() only supported for Byte or UInt16 bands in TIFF format.


0...10...20...30...40...50...60...70...80...90...100 - done.


In [ ]:
combine_rasters_with_gdal(output_pred_dir, output_mosaic)

# Binary Mosaic to Shapefile

In [68]:
import os
import numpy as np
from osgeo import gdal
from osgeo import ogr

def raster_to_vector(input_raster_path, output_vector_path):
    # Open the input raster
    raster = gdal.Open(input_raster_path)
    if raster is None:
        print(f"Unable to open input raster {input_raster_path}")
        return
    
    # Get raster information
    xsize = raster.RasterXSize
    ysize = raster.RasterYSize
    geotransform = raster.GetGeoTransform()
    projection = raster.GetProjection()
    
    # Create a new vector layer
    driver = ogr.GetDriverByName('ESRI Shapefile')
    output_vector = driver.CreateDataSource(output_vector_path)
    layer = output_vector.CreateLayer('output', srs=None)
    
    # Add a single field to the layer
    layer.CreateField(ogr.FieldDefn('value', ogr.OFTInteger))
    
    # Rasterize the vector layer
    gdal.RasterizeLayer(raster, [1], layer, options=['ATTRIBUTE=value'])
    
    # Close the vector layer
    output_vector = None


In [69]:
raster_to_vector(output_mosaic, output_shapefile)

ERROR 1: /Users/nirgut/Downloads/pipeline_test/combined_pred_new.tif: Write operation not permitted on dataset opened in read-only mode


# converting to polygons

In [71]:
st.pred2polygons(output_mosaic, output_shapefile)
st.del_0val_polygons(output_shapefile)